# 08. Robust Inference Strategy

Đánh giá chiến lược dùng PhoBERT mặc định và chuyển sang char SVM khi câu có khả năng thiếu dấu.

## 1. Setup

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown, Image

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd

print("Project root:", ROOT)

TABLES_DIR = ROOT / "outputs" / "tables"
FIGURES_DIR = ROOT / "outputs" / "figures"
REPORTS_DIR = ROOT / "outputs" / "reports"
METRICS_DIR = ROOT / "outputs" / "metrics"
PREDICTIONS_DIR = ROOT / "outputs" / "predictions"
NOISY_DIR = ROOT / "data" / "noisy"

## 2. Kiểm tra file Stage 8 bắt buộc

Các file này được tạo bởi hai lệnh:

```powershell
python scripts/generate_noisy_validation.py
python scripts/evaluate_robust_inference.py
```

In [ ]:
required_files = {
    "robust_config": ROOT / "configs" / "robust_inference.yaml",
    "robust_source": ROOT / "src" / "robust_inference.py",
    "generate_noisy_validation_script": ROOT / "scripts" / "generate_noisy_validation.py",
    "evaluate_robust_inference_script": ROOT / "scripts" / "evaluate_robust_inference.py",

    "validation_eval_all": NOISY_DIR / "validation_eval_all.csv",

    "noisy_validation_generation_summary": TABLES_DIR / "noisy_validation_generation_summary.csv",
    "noisy_validation_detection_thresholds": TABLES_DIR / "noisy_validation_detection_thresholds.csv",

    "selected_threshold": METRICS_DIR / "robust_inference_selected_threshold.json",

    "robust_inference_predictions": PREDICTIONS_DIR / "robust_inference_predictions.csv",
    "robust_inference_results": TABLES_DIR / "robust_inference_results.csv",
    "robust_inference_drop": TABLES_DIR / "robust_inference_drop.csv",
    "robust_inference_comparison": TABLES_DIR / "robust_inference_comparison.csv",
    "robust_inference_class_report": TABLES_DIR / "robust_inference_class_report.csv",
    "robust_inference_detector_overall": TABLES_DIR / "robust_inference_detector_overall.csv",
    "robust_inference_detector_by_variant": TABLES_DIR / "robust_inference_detector_by_variant.csv",

    "robust_inference_macro_f1_sentiment": FIGURES_DIR / "robust_inference_macro_f1_sentiment.png",
    "robust_inference_macro_f1_topic": FIGURES_DIR / "robust_inference_macro_f1_topic.png",
    "robust_inference_drop_sentiment": FIGURES_DIR / "robust_inference_drop_sentiment.png",
    "robust_inference_drop_topic": FIGURES_DIR / "robust_inference_drop_topic.png",

    "robust_inference_report": REPORTS_DIR / "robust_inference_report.md",
    "robust_inference_metrics": METRICS_DIR / "robust_inference_metrics.json",
}

check_df = pd.DataFrame(
    [{"name": name, "path": str(path), "exists": path.exists()} for name, path in required_files.items()]
)

display(check_df)

missing = check_df.loc[~check_df["exists"], "name"].tolist()
if missing:
    raise FileNotFoundError(f"Missing Stage 8 files: {missing}")

print("All required Stage 8 files exist.")

## 3. Load kết quả Stage 8

In [ ]:
validation_generation = pd.read_csv(TABLES_DIR / "noisy_validation_generation_summary.csv")
thresholds = pd.read_csv(TABLES_DIR / "noisy_validation_detection_thresholds.csv")
detector_overall = pd.read_csv(TABLES_DIR / "robust_inference_detector_overall.csv")
detector_by_variant = pd.read_csv(TABLES_DIR / "robust_inference_detector_by_variant.csv")

results = pd.read_csv(TABLES_DIR / "robust_inference_results.csv")
drop = pd.read_csv(TABLES_DIR / "robust_inference_drop.csv")
comparison = pd.read_csv(TABLES_DIR / "robust_inference_comparison.csv")
class_report = pd.read_csv(TABLES_DIR / "robust_inference_class_report.csv")

display(Markdown("### Noisy validation generation summary"))
display(validation_generation)

display(Markdown("### Detection threshold candidates"))
display(thresholds)

display(Markdown("### Detector overall"))
display(detector_overall)

display(Markdown("### Detector by variant"))
display(detector_by_variant)

display(Markdown("### Robust inference comparison preview"))
display(comparison.head(30))

## 4. Threshold selection on noisy validation

Nội dung: chọn ngưỡng phát hiện thiếu dấu trên **validation**, không dùng test để tối ưu.

Detector dùng đặc trưng:

```text
accented_ratio = số ký tự tiếng Việt có dấu / số ký tự alphabet
```

Nếu câu đủ dài và `accented_ratio` thấp hơn ngưỡng, hệ thống route sang TF-IDF char SVM.

In [ ]:
best_threshold = thresholds.sort_values(
    ["f1", "false_positive_rate", "threshold"],
    ascending=[False, True, True],
).iloc[0]

display(Markdown("### Selected threshold"))
display(best_threshold.to_frame().T)

display(Markdown(
    f"Selected threshold = **{best_threshold['threshold']}** "
    f"with detector F1 = **{best_threshold['f1']:.4f}**, "
    f"precision = **{best_threshold['precision']:.4f}**, "
    f"recall = **{best_threshold['recall']:.4f}**."
))

## 5. Detector behavior on test variants

Nội dung: kiểm tra detector có route đúng loại dữ liệu không.

Kỳ vọng:

```text
clean/noise nhẹ:
route_to_char_rate thấp

no_accent/mixed_no_accent:
route_to_char_rate cao
```

In [ ]:
detector_view = detector_by_variant.copy()
detector_view = detector_view.sort_values("variant")

display(detector_view)

for _, row in detector_view.iterrows():
    display(Markdown(
        f"- `{row['variant']}`: route_to_char_rate = "
        f"**{row['route_to_char_rate']:.4f}** "
        f"({int(row['route_to_char_svm'])}/{int(row['total'])} samples)."
    ))

## 6. Macro-F1 results by system

Nội dung: so sánh `phobert_only`, `tfidf_char_svm_only`, `robust_router`, `oracle_router`.

In [ ]:
variant_order = [
    "clean",
    "typo_light",
    "typo_medium",
    "teencode_light",
    "mixed_light",
    "no_accent",
    "mixed_no_accent",
]

system_order = [
    "phobert_only",
    "tfidf_char_svm_only",
    "robust_router",
    "oracle_router",
]

results_view = results.copy()
results_view["variant"] = pd.Categorical(results_view["variant"], categories=variant_order, ordered=True)
results_view["system"] = pd.Categorical(results_view["system"], categories=system_order, ordered=True)
results_view = results_view.sort_values(["task", "variant", "system"])

display(results_view[[
    "task",
    "variant",
    "system",
    "accuracy",
    "macro_f1",
    "weighted_f1",
    "num_samples",
]])

## 7. Sentiment: Robust Router có cải thiện không?

In [ ]:
sentiment = comparison[comparison["task"] == "sentiment"].copy()
sentiment["variant"] = pd.Categorical(sentiment["variant"], categories=variant_order, ordered=True)
sentiment["system"] = pd.Categorical(sentiment["system"], categories=system_order, ordered=True)
sentiment = sentiment.sort_values(["variant", "system"])

display(sentiment[[
    "variant",
    "system",
    "accuracy",
    "macro_f1",
    "macro_f1_gain_vs_phobert",
    "accuracy_gain_vs_phobert",
    "macro_f1_gap_to_oracle",
    "rank_macro_f1_within_variant",
]])

focus = sentiment[
    (sentiment["system"] == "robust_router") &
    (sentiment["variant"].isin(["no_accent", "mixed_no_accent"]))
]

for _, row in focus.iterrows():
    display(Markdown(
        f"- **Sentiment / {row['variant']}**: robust_router Macro-F1 = **{row['macro_f1']:.4f}**, "
        f"gain vs PhoBERT-only = **{row['macro_f1_gain_vs_phobert']:.4f}**, "
        f"accuracy gain = **{row['accuracy_gain_vs_phobert']:.4f}**."
    ))

## 8. Topic: Robust Router có cải thiện không?

In [ ]:
topic = comparison[comparison["task"] == "topic"].copy()
topic["variant"] = pd.Categorical(topic["variant"], categories=variant_order, ordered=True)
topic["system"] = pd.Categorical(topic["system"], categories=system_order, ordered=True)
topic = topic.sort_values(["variant", "system"])

display(topic[[
    "variant",
    "system",
    "accuracy",
    "macro_f1",
    "macro_f1_gain_vs_phobert",
    "accuracy_gain_vs_phobert",
    "macro_f1_gap_to_oracle",
    "rank_macro_f1_within_variant",
]])

focus = topic[
    (topic["system"] == "robust_router") &
    (topic["variant"].isin(["no_accent", "mixed_no_accent"]))
]

for _, row in focus.iterrows():
    display(Markdown(
        f"- **Topic / {row['variant']}**: robust_router Macro-F1 = **{row['macro_f1']:.4f}**, "
        f"gain vs PhoBERT-only = **{row['macro_f1_gain_vs_phobert']:.4f}**, "
        f"accuracy gain = **{row['accuracy_gain_vs_phobert']:.4f}**."
    ))

## 9. Macro-F1 visualization

In [ ]:
display(Markdown("### Sentiment Macro-F1"))
display(Image(filename=str(FIGURES_DIR / "robust_inference_macro_f1_sentiment.png")))

display(Markdown("### Topic Macro-F1"))
display(Image(filename=str(FIGURES_DIR / "robust_inference_macro_f1_topic.png")))

## 10. Macro-F1 drop visualization

In [ ]:
display(Markdown("### Sentiment Macro-F1 drop"))
display(Image(filename=str(FIGURES_DIR / "robust_inference_drop_sentiment.png")))

display(Markdown("### Topic Macro-F1 drop"))
display(Image(filename=str(FIGURES_DIR / "robust_inference_drop_topic.png")))

## 11. Clean/noise nhẹ có bị ảnh hưởng không?

Nội dung: kiểm tra robust_router có làm giảm hiệu năng trên clean/noise nhẹ hay không.

In [ ]:
light_variants = ["clean", "typo_light", "typo_medium", "teencode_light", "mixed_light"]

light_check = comparison[
    (comparison["system"] == "robust_router") &
    (comparison["variant"].isin(light_variants))
].copy()

light_check = light_check.sort_values(["task", "variant"])

display(light_check[[
    "task",
    "variant",
    "macro_f1",
    "macro_f1_gain_vs_phobert",
    "accuracy",
    "accuracy_gain_vs_phobert",
]])

max_abs_loss = (
    light_check.assign(loss_vs_phobert=-light_check["macro_f1_gain_vs_phobert"])
    .sort_values("loss_vs_phobert", ascending=False)
    .head(1)
)

display(Markdown("### Largest Macro-F1 loss vs PhoBERT-only on clean/light variants"))
display(max_abs_loss[[
    "task",
    "variant",
    "macro_f1_gain_vs_phobert",
    "accuracy_gain_vs_phobert",
]])

## 12. No-accent improvement summary

Nội dung: tạo bảng ngắn dùng trực tiếp trong báo cáo.

In [ ]:
no_accent_focus = comparison[
    (comparison["system"] == "robust_router") &
    (comparison["variant"].isin(["no_accent", "mixed_no_accent"]))
].copy()

summary = no_accent_focus[[
    "task",
    "variant",
    "macro_f1",
    "phobert_only_macro_f1",
    "macro_f1_gain_vs_phobert",
    "accuracy",
    "phobert_only_accuracy",
    "accuracy_gain_vs_phobert",
    "macro_f1_gap_to_oracle",
]].sort_values(["task", "variant"])

display(summary)

## 13. Per-class analysis

Nội dung: xem Robust Router thay đổi F1 theo từng lớp như thế nào.

In [ ]:
summary_labels = {"accuracy", "macro avg", "weighted avg"}
pc = class_report[~class_report["label"].isin(summary_labels)].copy()

focus_pc = pc[
    (pc["system"].isin(["phobert_only", "robust_router"])) &
    (pc["variant"].isin(["clean", "no_accent", "mixed_no_accent"]))
].copy()

for task in focus_pc["task"].dropna().unique():
    display(Markdown(f"### Per-class F1 — {task}"))
    task_df = focus_pc[focus_pc["task"] == task]
    pivot = task_df.pivot_table(
        index=["variant", "label"],
        columns="system",
        values="f1_score",
        observed=False,
    ).reset_index()
    if "phobert_only" in pivot.columns and "robust_router" in pivot.columns:
        pivot["router_minus_phobert"] = pivot["robust_router"] - pivot["phobert_only"]
    display(pivot)

## 14. Kết luận Stage 8

Kết luận rút ra từ kết quả:

```text
1. Detector thiếu dấu được chọn ngưỡng trên noisy validation, không dùng noisy test.
2. Detector route rất ít mẫu clean/noise nhẹ sang char SVM.
3. Detector route phần lớn no_accent/mixed_no_accent sang char SVM.
4. Robust Router gần như giữ nguyên hiệu năng của PhoBERT trên clean/noise nhẹ.
5. Robust Router cải thiện rõ trên no_accent và mixed_no_accent.
6. Stage 8 cải thiện hệ thống suy luận, không cải thiện bản thân PhoBERT.
```

Câu diễn giải an toàn cho báo cáo:

```text
Chiến lược robust inference sử dụng PhoBERT làm mô hình mặc định và chuyển sang TF-IDF char SVM khi đầu vào có khả năng thiếu dấu. Kết quả cho thấy cách kết hợp này giữ được hiệu năng cao của PhoBERT trên dữ liệu chuẩn và nhiễu nhẹ, đồng thời cải thiện đáng kể trên các biến thể thiếu dấu.
```

## 15. Stage 8 status

Stage 8 hoàn thành khi có đủ:

```text
configs/robust_inference.yaml
src/robust_inference.py
scripts/generate_noisy_validation.py
scripts/evaluate_robust_inference.py

data/noisy/validation_eval_all.csv

outputs/tables/noisy_validation_generation_summary.csv
outputs/tables/noisy_validation_detection_thresholds.csv
outputs/tables/robust_inference_detector_overall.csv
outputs/tables/robust_inference_detector_by_variant.csv
outputs/tables/robust_inference_results.csv
outputs/tables/robust_inference_drop.csv
outputs/tables/robust_inference_comparison.csv
outputs/tables/robust_inference_class_report.csv

outputs/predictions/robust_inference_predictions.csv

outputs/figures/robust_inference_macro_f1_sentiment.png
outputs/figures/robust_inference_macro_f1_topic.png
outputs/figures/robust_inference_drop_sentiment.png
outputs/figures/robust_inference_drop_topic.png

outputs/reports/robust_inference_report.md
outputs/metrics/robust_inference_selected_threshold.json
outputs/metrics/robust_inference_metrics.json
```

Sau Stage 8, dự án đủ điều kiện chuyển sang viết báo cáo chính.